In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
from pyhessian import hessian
from tqdm import tqdm
import pandas as pd
import os
import matplotlib.pyplot as plt

from train_mlp import muMLPTab9

device = "cuda"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:
def get_cifar(batch_size=128, num_classes=10, subset=1.0, MSE=False, one_hot=False, on_gpu=False, device=None, download=False):
    assert 10 >= num_classes, f"Number of classes {np.unique(targets[indices]).shape[0]} != {num_classes}"
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])
    
    train_ds = datasets.CIFAR10(root='/tmp', train=True, download=download, transform=transform)
    targets = np.array(train_ds.targets)
    mask = np.isin(targets, np.arange(num_classes))

    if subset < 1.0:
        indices = []
        samples_per_class = int(5000 * subset)
        for i in range(num_classes):
            class_indices = np.where(targets == i)[0]
            selected_indices = np.random.choice(class_indices, samples_per_class, replace=False)
            indices.extend(selected_indices)
    else:
        indices = np.where(mask)[0]

    X, y = [], []
    for i in tqdm(indices):
        x, y_ = train_ds[i]
        X.append(x)
        y.append(y_)
    X = torch.stack(X)
    y = torch.tensor(y)

    if MSE and one_hot:
        y = F.one_hot(y, num_classes=num_classes).float()
    elif MSE and num_classes == 2:
        y = torch.where(y == 1, torch.tensor(1.0), torch.tensor(-1.0))
    elif MSE:
        raise ValueError("MSE is only supported for binary classification or one-hot encoding.")

    if on_gpu:
        assert device is not None, "Please provide a device="
        X = X.to(device)
        y = y.to(device)

    tensor_ds = TensorDataset(X, y)
    train_dl = DataLoader(tensor_ds, batch_size=batch_size, shuffle=True, pin_memory=not on_gpu)

    if on_gpu:
        print(f"Estimated size of the dataset in MB: {(X.numel() * X.element_size() + y.numel() * y.element_size()) / 1024 / 1024:.2f}")

    return train_dl, tensor_ds


In [3]:
seed = 1
epochs = 5
classes = 2

# Tensors loaded on GPU per batch

In [4]:
dl, ds = get_cifar(batch_size=128, num_classes=classes, MSE=False, on_gpu=False)
print(len(dl))

torch.manual_seed(seed)
np.random.seed(seed)
print(next(iter(dl))[1].shape)
model = muMLPTab9(128, classes).to(device)
criterion = nn.CrossEntropyLoss()

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
for epoch in range(epochs):
    epoch_loss = 0
    for i, (X, y) in enumerate(dl):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X.size(0)
    print(epoch_loss / len(dl.dataset))

100%|██████████| 10000/10000 [00:02<00:00, 4073.63it/s]


79
torch.Size([128])
0.6877436098098755
0.6252798287391662
0.5944725264549255
0.5714316897392273
0.553438679933548


In [44]:
torch.manual_seed(seed)
np.random.seed(seed)
print(next(iter(dl))[1].shape)
model = muMLPTab9(128, classes).to(device)
criterion = nn.CrossEntropyLoss()

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
for epoch in range(epochs):
    epoch_loss = 0
    for i, (X, y) in enumerate(dl):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X.size(0)
    print(epoch_loss / len(dl.dataset))

torch.Size([128])
0.6877436098098755
0.6252798287391662
0.5944725264549255
0.5714316897392273
0.553438679933548


# Tensors on GPU

In [45]:
dl, ds = get_cifar(batch_size=128, num_classes=classes, MSE=False, on_gpu=True, device=device)
print(len(dl))

torch.manual_seed(seed)
np.random.seed(seed)
print(next(iter(dl))[1].shape)
model = muMLPTab9(128, classes).to(device)
criterion = nn.CrossEntropyLoss()

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
for epoch in range(epochs):
    epoch_loss = 0
    for i, (X, y) in enumerate(dl):
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X.size(0)
    print(epoch_loss / len(dl.dataset))

100%|██████████| 10000/10000 [00:01<00:00, 6591.93it/s]


Estimated size of the dataset in MB: 117.26
79
torch.Size([128])
0.6877436098098755
0.6252798287391662
0.5944725264549255
0.5714316897392273
0.553438679933548


In [46]:
torch.manual_seed(seed)
np.random.seed(seed)
print(next(iter(dl))[1].shape)
model = muMLPTab9(128, classes).to(device)
criterion = nn.CrossEntropyLoss()

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
for epoch in range(epochs):
    epoch_loss = 0
    for i, (X, y) in enumerate(dl):
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X.size(0)
    print(epoch_loss / len(dl.dataset))

torch.Size([128])
0.6877436098098755
0.6252798287391662
0.5944725264549255
0.5714316897392273
0.553438679933548


# MSE + on GPU

In [10]:
dl, ds = get_cifar(batch_size=128, num_classes=classes, MSE=True, on_gpu=True, device=device)
print(len(dl))

torch.manual_seed(seed)
np.random.seed(seed)
print(next(iter(dl))[1].shape)
model = muMLPTab9(128, classes).to(device)
criterion = nn.MSELoss()

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
for epoch in range(epochs):
    epoch_loss = 0
    for i, (X, y) in enumerate(dl):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X.size(0)
    print(epoch_loss / len(dl.dataset))

100%|██████████| 10000/10000 [00:02<00:00, 4587.80it/s]


Estimated size of the dataset in MB: 117.26
79
torch.Size([128, 2])
0.6868212818145752
0.4809396454811096
0.4108572193145752
0.36817205924987795
0.33871090376377105


In [48]:
torch.manual_seed(seed)
np.random.seed(seed)
print(next(iter(dl))[1].shape)
model = muMLPTab9(128, classes).to(device)
criterion = nn.MSELoss()

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
for epoch in range(epochs):
    epoch_loss = 0
    for i, (X, y) in enumerate(dl):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X.size(0)
    print(epoch_loss / len(dl.dataset))

torch.Size([128, 2])
0.6868212818145752
0.4809396454811096
0.4108572193145752


0.36817205924987795
0.33871090376377105


# MSE but -1, 1

In [6]:
class muMLPTab9(nn.Module):
    """muP initialized MLP model, according to Table9 from TP5 (thanks to dvruette)"""
    def __init__(self, width=128, num_classes=10, activation="relu"):
        super().__init__()
        self.width = width
        self.input_mult = self.width**0.5
        self.output_mult = self.width**-0.5
        self.act = nn.ReLU() if activation == "relu" else nn.ELU()
        self.fc_1 = nn.Linear(3072, width, bias=False)
        self.fc_2 = nn.Linear(width, width, bias=False)
        self.fc_3 = nn.Linear(width, num_classes, bias=False)
        self.reset_parameters()

    def get_name(self):
        return "mup"
    
    def reset_parameters(self):
        nn.init.normal_(self.fc_1.weight, std=self.width**-0.5*2/np.sqrt(3072)) # ? 1/fanout 
        # nn.init.normal_(self.fc_1.weight, std=self.width**-0.5)
        nn.init.normal_(self.fc_2.weight, std=self.width**-0.5)
        nn.init.normal_(self.fc_3.weight, std=self.width**-0.5)

    def forward(self, x, record_activations=False):
        if record_activations:
            activations = []
            x = x.view(x.size(0), -1)
            h = self.input_mult * self.fc_1(x)
            activations.append(h)
            h = self.fc_2(F.relu(h))
            activations.append(h)
            h = self.output_mult * self.fc_3(F.relu(h))
            activations.append(h)
            return h, activations
        
        x = x.view(x.size(0), -1)
        h = self.input_mult * self.fc_1(x)
        h = self.fc_2(F.relu(h))
        h = self.output_mult * self.fc_3(F.relu(h))
        return h

In [ ]:
import torch
import torch.nn as nn
import torch.nn.init as init

class FullyConnectedELUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(3072, 200, bias=False)
        self.fc2 = nn.Linear(200, 200, bias=False)
        self.fc3 = nn.Linear(200, 200, bias=False)
        self.fc4 = nn.Linear(200, 200, bias=False)
        self.fc5 = nn.Linear(200, 1, bias=False)
        self.activation = nn.ELU()
        self._init_weights()

    def _init_weights(self):
        for layer in [self.fc1, self.fc2, self.fc3, self.fc4, self.fc5]:
            init.xavier_uniform_(layer.weight, gain=1.0)

    def forward(self, x):
        x = x.view(x.size(0), -1)   # Flatten (B, 3, 32, 32) → (B, 3072)
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.activation(self.fc3(x))
        x = self.activation(self.fc4(x))
        x = self.fc5(x)             # Output: shape (B, 1), no activation
        return x

dl, ds = get_cifar(batch_size=128, num_classes=classes, MSE=True, one_hot=False, on_gpu=True, device=device, subset=0.005)
print(len(dl))

torch.manual_seed(seed)
np.random.seed(seed)
print(next(iter(dl))[1].shape)
# model = muMLPTab9(4096, classes-1, activation="elu").to(device)
model = FullyConnectedELUNet().to(device)
criterion = nn.MSELoss()

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
for epoch in range(200):
    epoch_loss = 0
    for i, (X, y) in enumerate(dl):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X.size(0)
    # grad_means = {
    #     "fc_1": model.fc_1.weight.grad.mean().item(),
    #     "fc_2": model.fc_2.weight.grad.mean().item(),
    #     "fc_3": model.fc_3.weight.grad.mean().item(),
    # }
    # print(f"Epoch {epoch}: Model Gradients - {grad_means}")
    print(epoch_loss / len(dl.dataset))

In [23]:
dl, ds = get_cifar(batch_size=128, num_classes=classes, MSE=True, one_hot=False, on_gpu=True, device=device, subset=0.005)
print(len(dl))

torch.manual_seed(seed)
np.random.seed(seed)
print(next(iter(dl))[1].shape)
# model = muMLPTab9(4096, classes-1, activation="elu").to(device)
model = FullyConnectedELUNet().to(device)
criterion = nn.MSELoss()

model.train()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
for epoch in range(200):
    epoch_loss = 0
    for i, (X, y) in enumerate(dl):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X.size(0)
    # grad_means = {
    #     "fc_1": model.fc_1.weight.grad.mean().item(),
    #     "fc_2": model.fc_2.weight.grad.mean().item(),
    #     "fc_3": model.fc_3.weight.grad.mean().item(),
    # }
    # print(f"Epoch {epoch}: Model Gradients - {grad_means}")
    print(epoch_loss / len(dl.dataset))

100%|██████████| 50/50 [00:00<00:00, 4171.61it/s]


Estimated size of the dataset in MB: 0.59
1
torch.Size([50])
1.284411072731018
1.2782366275787354
1.0900702476501465
1.1302940845489502
1.1269614696502686
1.106525182723999
1.049023985862732
1.0780072212219238
1.0416849851608276
1.056017518043518
1.0432462692260742
1.0213226079940796
1.0342429876327515
1.0287472009658813
1.0148893594741821
1.021836519241333
1.0247242450714111
1.0127595663070679
1.007187008857727
1.01161789894104
1.0123661756515503
1.0072963237762451
1.0048962831497192
1.0069068670272827
1.0077577829360962
1.005821943283081
1.0045123100280762
1.0049251317977905
1.0048991441726685
1.0037070512771606
1.0028717517852783
1.0029869079589844
1.002925157546997
1.0021655559539795
1.0016320943832397
1.0019054412841797
1.0022368431091309
1.0018473863601685
1.001114845275879
1.0008643865585327
1.0011284351348877
1.001238465309143
1.0008845329284668
1.0004931688308716
1.000511646270752
1.0007811784744263
1.0008405447006226
1.0005829334259033
1.0003252029418945
1.0003153085708618
1.

In [ ]:
# Minimal working reproduction
import torch, torchvision, itertools, math
from torch import nn
from torchvision.transforms import ToTensor
from pyhessian import hessian

torch.manual_seed(0)
torch.set_default_dtype(torch.float64)           # (optional but faithful)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. 50-sample binary classification subset of CIFAR-10
full = torchvision.datasets.CIFAR10(".", train=True, download=True, transform=ToTensor())
idx0 = [i for i,t in enumerate(full.targets) if t == 0][:25]   # airplane → +1
idx1 = [i for i,t in enumerate(full.targets) if t == 6][:25]   # frog     → –1
x = torch.stack([full[i][0] for i in idx0+idx1]).to(device)
y = torch.tensor([ 1.] * 25 + [-1.] * 25, device=device).unsqueeze(1)
loader = itertools.cycle([(x, y)])

dataloader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x, y), batch_size=50, shuffle=False)

# 2. 5-layer ELU FC network (bias=False everywhere, as in the paper)
class FCELU(nn.Module):
    def __init__(self):
        super().__init__()
        layers = [nn.Flatten()]
        for _ in range(4):
            layers += [nn.Linear(3072 if _ == 0 else 5000, 5000, bias=False), nn.ELU()]
        layers += [nn.Linear(5000, 1, bias=False)]
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=1)  # ELU gain

    def forward(self, z): return self.net(z)

lr = 0.001
model   = FCELU().to(device)
# model = muMLPTab9(256, 1).to(device)
opt     = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.0)  # η = 0.01 ⇒ 2/η = 200
loss_f  = nn.MSELoss()
criterion = nn.MSELoss()

# 3. Gradient-descent loop (18 500 updates = paper’s x-axis)
eigens = []
print("EOS:", 2/lr)
for step in range(18_500):
    xb, yb = next(loader)
    opt.zero_grad()
    loss = loss_f(model(xb), yb)
    loss.backward()
    opt.step()
    if step % 100 == 0:
        hess = hessian(model, criterion=criterion, data=(x, y), cuda=True)
        top_eigenvalue = hess.eigenvalues(maxIter=400, top_n=1)[0][0]
        eigens.append(top_eigenvalue)
        print(f"{step:5d}  loss {loss.item():.8}  eig {top_eigenvalue:.8}")
    # if step % 1000 == 0:
    #     print(f"{step:5d}  loss {loss.item():.8}")   # slides from ≈1 → ≈1e-7

# lr = 0.01
# model = muMLPTab9(512, 1).to(device)
# opt     = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.0)  # η = 0.01 ⇒ 2/η = 200
# loss_f  = nn.MSELoss()
# eigens = []
# print("EOS:", 2/lr)
# for epoch in range(18000):
#     for step, (xb, yb) in enumerate(dataloader):
#         # xb, yb = next(loader)
#         opt.zero_grad()
#         loss = loss_f(model(xb), yb)
#         loss.backward()
#         opt.step()
#         if epoch % 500 == 0:
#             hess = hessian(model, criterion=criterion, data=(xb, yb), cuda=True)
#             top_eigenvalue = hess.eigenvalues(maxIter=400, top_n=1)[0][0]
#             eigens.append(top_eigenvalue)
#             print(f"{epoch:5d}  loss {loss.item():.8}  eig {top_eigenvalue:.8}")
#         # if step % 1000 == 0:
#         #     print(f"{step:5d}  loss {loss.item():.8}")   # slides from ≈1 → ≈1e-7


EOS: 2000.0
    0  loss 1.0096298  eig 6463.0658
  100  loss nan  eig nan
  200  loss nan  eig nan
  300  loss nan  eig nan
  400  loss nan  eig nan
  500  loss nan  eig nan
  600  loss nan  eig nan
  700  loss nan  eig nan
  800  loss nan  eig nan
  900  loss nan  eig nan
 1000  loss nan  eig nan
 1100  loss nan  eig nan
 1200  loss nan  eig nan


In [5]:
# Minimal working reproduction
import torch, torchvision, itertools, math
from torch import nn
from torchvision.transforms import ToTensor

torch.manual_seed(0)
torch.set_default_dtype(torch.float16)           # (optional but faithful)
device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. 50-sample binary classification subset of CIFAR-10
full = torchvision.datasets.CIFAR10(".", train=True, download=True, transform=ToTensor())
idx0 = [i for i,t in enumerate(full.targets) if t == 0][:25]   # airplane → +1
idx1 = [i for i,t in enumerate(full.targets) if t == 6][:25]   # frog     → –1
x = torch.stack([full[i][0] for i in idx0+idx1]).to(device)
y = torch.tensor([ 1.] * 25 + [-1.] * 25, device=device).unsqueeze(1)
loader = itertools.cycle([(x, y)])

dataloader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(x, y), batch_size=50, shuffle=False)

lr = 0.01
model = muMLPTab9(512, 1).to(device)
opt     = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.0)  # η = 0.01 ⇒ 2/η = 200
loss_f  = nn.MSELoss()
eigens = []
print("EOS:", 2/lr)
for epoch in range(18000):
    for step, (xb, yb) in enumerate(dataloader):
        # xb, yb = next(loader)
        opt.zero_grad()
        loss = loss_f(model(xb), yb)
        loss.backward()
        opt.step()
        if epoch % 500 == 0:
            hess = hessian(model, criterion=loss_f, data=(xb, yb), cuda=True)
            top_eigenvalue = hess.eigenvalues(maxIter=400, top_n=1)[0][0]
            eigens.append(top_eigenvalue)
            print(f"{epoch:5d}  loss {loss.item():.8}  eig {top_eigenvalue:.8}")
        # if step % 1000 == 0:
        #     print(f"{step:5d}  loss {loss.item():.8}")   # slides from ≈1 → ≈1e-7


EOS: 200.0
    0  loss 0.99755859  eig 0.0
  500  loss 0.010414124  eig 0.0
 1000  loss 0.00023651123  eig 0.0
 1500  loss 8.2612038e-05  eig 0.0
 2000  loss 4.4047832e-05  eig 0.0
 2500  loss 2.8192997e-05  eig 0.0
 3000  loss 1.9550323e-05  eig 0.0
 3500  loss 1.4543533e-05  eig 0.0
 4000  loss 1.0728836e-05  eig 0.0
 4500  loss 9.4771385e-06  eig 0.0


KeyboardInterrupt: 